# Steane7 Pauli frame correction check

This notebook checks the last two lines of `steane7_initialize` in `python/bloqade/lanes/arch/gemini/logical/upstream.py`:

```python
squin.x(qubits[3])
squin.broadcast.z([qubits[1], qubits[5]])
```

The test strategy is:

1. factor the encoder into `prepare input`, `native encoder core`, and `Pauli frame correction`;
2. check Steane stabilizer and logical-observable expectations before and after the correction;
3. call `steane7_initialize` directly on a few Clifford input states and verify the same logical signatures.


In [1]:
import math

import stim

from bloqade import squin
from bloqade.squin import kernel
from bloqade.tsim import Circuit
from bloqade.lanes.arch.gemini.logical.upstream import steane7_initialize


We will check the six standard Steane stabilizers and the current logical observables used in this repo:

- X stabilizers on supports `0123`, `1245`, `2346`
- Z stabilizers on the same supports
- logical X and Z on support `015`

For a correct encoded state, all six stabilizers should have expectation `+1`. The logical expectation should depend only on the input state.


In [2]:
STEANE_X_STABILIZERS = (
    stim.PauliString("+XXXX___"),
    stim.PauliString("+_XX_XX_"),
    stim.PauliString("+__XXX_X"),
)

STEANE_Z_STABILIZERS = (
    stim.PauliString("+ZZZZ___"),
    stim.PauliString("+_ZZ_ZZ_"),
    stim.PauliString("+__ZZZ_Z"),
)

LOGICAL_X = stim.PauliString("+XX___X_")
LOGICAL_Z = stim.PauliString("+ZZ___Z_")


def steane_signature(method):
    sim = stim.TableauSimulator()
    sim.do_circuit(stim.Circuit(str(Circuit(method))))
    return {
        "x_stabs": tuple(
            int(sim.peek_observable_expectation(p)) for p in STEANE_X_STABILIZERS
        ),
        "z_stabs": tuple(
            int(sim.peek_observable_expectation(p)) for p in STEANE_Z_STABILIZERS
        ),
        "logical_x": int(sim.peek_observable_expectation(LOGICAL_X)),
        "logical_z": int(sim.peek_observable_expectation(LOGICAL_Z)),
    }


def pretty_row(label, variant, sig):
    return {
        "input": label,
        "variant": variant,
        "X stabilizers": sig["x_stabs"],
        "Z stabilizers": sig["z_stabs"],
        "Lx": sig["logical_x"],
        "Lz": sig["logical_z"],
    }


To keep the comparison purely Clifford, we factor out the initial `u3(...)` and work with Clifford preparations on the data qubit `q[6]`. That isolates exactly what the final `X_3 Z_1 Z_5` frame update is doing.


In [3]:
@kernel
def prepare_zero(qubit):
    return


@kernel
def prepare_one(qubit):
    squin.x(qubit)


@kernel
def prepare_plus(qubit):
    squin.h(qubit)


@kernel
def prepare_minus(qubit):
    squin.x(qubit)
    squin.h(qubit)


PREPARE_BY_LABEL = {
    "|0>": prepare_zero,
    "|1>": prepare_one,
    "|+>": prepare_plus,
    "|->": prepare_minus,
}


@kernel
def native_encoder_core_without_frame(q):
    squin.broadcast.sqrt_y_adj(q[:6])
    evens = q[::2]
    odds = q[1::2]

    squin.broadcast.cz(odds, evens[1:])
    squin.sqrt_y(q[6])
    squin.broadcast.cz(evens[:-1], [q[3], q[5], q[6]])
    squin.broadcast.sqrt_y(q[2:])
    squin.broadcast.cz(evens[:-1], odds)
    squin.broadcast.sqrt_y([q[1], q[2], q[4]])


@kernel
def pauli_frame_correction(q):
    squin.x(q[3])
    squin.broadcast.z([q[1], q[5]])


def make_factored_kernel(prepare, *, with_frame):
    @kernel
    def circuit():
        q = squin.qalloc(7)
        prepare(q[6])
        native_encoder_core_without_frame(q)
        if with_frame:
            pauli_frame_correction(q)

    return circuit


In [4]:
factored_results = []

for label, prepare in PREPARE_BY_LABEL.items():
    no_frame = steane_signature(make_factored_kernel(prepare, with_frame=False))
    with_frame = steane_signature(make_factored_kernel(prepare, with_frame=True))

    factored_results.append(
        pretty_row(label, "without frame correction", no_frame)
    )
    factored_results.append(
        pretty_row(label, "with X3 Z1 Z5 frame correction", with_frame)
    )

factored_results


[{'input': '|0>',
  'variant': 'without frame correction',
  'X stabilizers': (-1, 1, 1),
  'Z stabilizers': (-1, 1, -1),
  'Lx': 0,
  'Lz': 1},
 {'input': '|0>',
  'variant': 'with X3 Z1 Z5 frame correction',
  'X stabilizers': (1, 1, 1),
  'Z stabilizers': (1, 1, 1),
  'Lx': 0,
  'Lz': 1},
 {'input': '|1>',
  'variant': 'without frame correction',
  'X stabilizers': (-1, 1, 1),
  'Z stabilizers': (-1, 1, -1),
  'Lx': 0,
  'Lz': -1},
 {'input': '|1>',
  'variant': 'with X3 Z1 Z5 frame correction',
  'X stabilizers': (1, 1, 1),
  'Z stabilizers': (1, 1, 1),
  'Lx': 0,
  'Lz': -1},
 {'input': '|+>',
  'variant': 'without frame correction',
  'X stabilizers': (-1, 1, 1),
  'Z stabilizers': (-1, 1, -1),
  'Lx': 1,
  'Lz': 0},
 {'input': '|+>',
  'variant': 'with X3 Z1 Z5 frame correction',
  'X stabilizers': (1, 1, 1),
  'Z stabilizers': (1, 1, 1),
  'Lx': 1,
  'Lz': 0},
 {'input': '|->',
  'variant': 'without frame correction',
  'X stabilizers': (-1, 1, 1),
  'Z stabilizers': (-1, 1, -1

The pattern to look for is:

- without frame correction, the logical observable is already correct, but the first X stabilizer and two Z stabilizers have sign `-1`;
- after applying `X_3 Z_1 Z_5`, all six Steane stabilizers become `+1`;
- the logical eigenvalue stays unchanged.

That is exactly the behavior we want from a Pauli frame correction.


In [5]:
for label, prepare in PREPARE_BY_LABEL.items():
    no_frame = steane_signature(make_factored_kernel(prepare, with_frame=False))
    with_frame = steane_signature(make_factored_kernel(prepare, with_frame=True))

    assert no_frame["x_stabs"] == (-1, 1, 1), label
    assert no_frame["z_stabs"] == (-1, 1, -1), label
    assert with_frame["x_stabs"] == (1, 1, 1), label
    assert with_frame["z_stabs"] == (1, 1, 1), label
    assert with_frame["logical_x"] == no_frame["logical_x"], label
    assert with_frame["logical_z"] == no_frame["logical_z"], label

print("Factored frame-correction checks passed.")


Factored frame-correction checks passed.


Now we connect back to the actual implementation by calling `steane7_initialize(...)` directly on four Clifford input states. The `u3` angles below prepare `|0>`, `|1>`, `|+>`, and `|->` on the input qubit before encoding.


In [6]:
ANGLE_BY_LABEL = {
    "|0>": (0.0, 0.0, 0.0),
    "|1>": (math.pi, 0.0, 0.0),
    "|+>": (math.pi / 2, 0.0, 0.0),
    "|->": (math.pi / 2, math.pi, 0.0),
}


def make_actual_initializer_kernel(label):
    theta, phi, lam = ANGLE_BY_LABEL[label]

    @kernel
    def circuit():
        q = squin.qalloc(7)
        steane7_initialize(theta, phi, lam, q)

    return circuit


actual_results = []

for label in ANGLE_BY_LABEL:
    sig = steane_signature(make_actual_initializer_kernel(label))
    actual_results.append(pretty_row(label, "direct steane7_initialize", sig))

actual_results


[{'input': '|0>',
  'variant': 'direct steane7_initialize',
  'X stabilizers': (1, 1, 1),
  'Z stabilizers': (1, 1, 1),
  'Lx': 0,
  'Lz': 1},
 {'input': '|1>',
  'variant': 'direct steane7_initialize',
  'X stabilizers': (1, 1, 1),
  'Z stabilizers': (1, 1, 1),
  'Lx': 0,
  'Lz': -1},
 {'input': '|+>',
  'variant': 'direct steane7_initialize',
  'X stabilizers': (1, 1, 1),
  'Z stabilizers': (1, 1, 1),
  'Lx': 1,
  'Lz': 0},
 {'input': '|->',
  'variant': 'direct steane7_initialize',
  'X stabilizers': (1, 1, 1),
  'Z stabilizers': (1, 1, 1),
  'Lx': -1,
  'Lz': 0}]

In [7]:
expected_logical = {
    "|0>": {"logical_x": 0, "logical_z": 1},
    "|1>": {"logical_x": 0, "logical_z": -1},
    "|+>": {"logical_x": 1, "logical_z": 0},
    "|->": {"logical_x": -1, "logical_z": 0},
}

for label, expected in expected_logical.items():
    sig = steane_signature(make_actual_initializer_kernel(label))
    assert sig["x_stabs"] == (1, 1, 1), label
    assert sig["z_stabs"] == (1, 1, 1), label
    assert sig["logical_x"] == expected["logical_x"], label
    assert sig["logical_z"] == expected["logical_z"], label

print("Direct steane7_initialize checks passed.")


Direct steane7_initialize checks passed.


Conclusion:

- `X(q[3])` and `Z(q[1]), Z(q[5])` do act as a Pauli frame correction;
- they fix the stabilizer signs introduced by the native encoder core;
- they do **not** change the encoded logical information.

So the correction is doing the right thing: it moves the output from the correct logical sector with a bad frame into the standard Steane code space with the expected `+1` stabilizers.
